# Text-to-SQL — Gemini (snapshot-only)
Clone of text_to_sql_self_healthy_kafka.ipynb using the shared analytics runtime.
Both SQL and Vietnamese narration use **gemini-3.5-flash**, official Google Gemini API.
No HF/Ollama calls. Run cells in order; Step A and Step B have separate timings.

## Configuration
Install from a terminal at the repository root: `python -m pip install -r notebooks/gemini/requirements.txt`.
Set GEMINI_API_KEY outside source in ignored `.env.gemini` or process environment.
GEMINI_MODEL_ID defaults to `gemini-3.5-flash` (not `gemini-flash-3.5`).
Optional: GEMINI_THINKING_LEVEL / GEMINI_RESPONSE_THINKING_LEVEL (default low;
minimal/low/medium/high/default), GEMINI_REQUEST_TIMEOUT_SECONDS (60),
GEMINI_SQL_MAX_TOKENS / GEMINI_RESPONSE_MAX_TOKENS (4096 each).
Thinking is model-dependent; unsupported settings fail explicitly, never switch providers.

This notebook NEVER refreshes the snapshot. BENCHMARK_SQLITE_PATH defaults to
self_healthy_kafka_snapshot.db. Source refresh remains a deliberate separate action in
the original notebook; do not run it during provider comparisons.
Raw Message/Details columns are excluded and blocked by the SQLite authorizer.
Only approved schema, categorical metadata and bounded query results are sent.
Structured JSON and successful SQL execution do not prove semantic accuracy.

## Compare
Offline: `python notebooks/evaluation/evaluate.py --backend gemini`
Live (billable): add `--live`. Use `--fixture normal` for the isolated duration fixture.
HF evaluator remains the default; use the same unmodified snapshot and cases.
Never save API keys in notebook outputs. Provider runtime flows are preserved; notebook paths and imports have been reorganized.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "notebooks" / "shared" / "analytics.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the self-healthy-kafka repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env.gemini")
load_dotenv(REPO_ROOT / ".env")

In [ ]:
import importlib

import notebooks.gemini.adapter as notebook_gemini
import notebooks.shared.analytics as notebook_analytics
import notebooks.shared.diagnostics as notebook_diagnostics
import notebooks.shared.few_shot as notebook_few_shot
import notebooks.shared.semantics as notebook_semantics

for module in (
    notebook_semantics,
    notebook_diagnostics,
    notebook_few_shot,
    notebook_analytics,
    notebook_gemini,
):
    importlib.reload(module)
from notebooks.gemini.adapter import make_gemini_workflow  # noqa: E402 - reload first

In [ ]:
import json

snapshot_path = Path(os.getenv("BENCHMARK_SQLITE_PATH", "self_healthy_kafka_snapshot.db"))
if not snapshot_path.is_absolute():
    snapshot_path = REPO_ROOT / snapshot_path
# Missing credentials fail here. No refresh or provider fallback.
workflow = make_gemini_workflow(snapshot_path)
snapshot = workflow.snapshot
print(
    json.dumps(
        {
            "model": workflow.model_id,
            "provider": workflow.provider,
            "snapshot": snapshot.metadata(),
            "thinking": workflow.client.thinking,
        },
        ensure_ascii=False,
        indent=2,
    )
)

## Step A — Gemini sinh SQL + SQLite thực thi
Chỉnh question rồi chạy cell này; tối đa 3 model calls, không có retry ngầm.

In [ ]:
import json

question = "Thời gian xử lý trung bình (tính bằng phút) từ lúc nhận (ReceivedAt) đến khi hoàn tất (CompletedAt) của các queue thành công là bao nhiêu?"

verified_result = None
final_answer = None
workflow.reset()
try:
    verified_result = workflow.query(question)
    if workflow.clarification:
        print("Clarification required:", workflow.clarification)
    else:
        print("Verified returned rows:", verified_result["returned_row_count"])
        print("Truncated:", verified_result["truncated"])
        print("Model interpretation (not independently verified):", workflow.interpretation)
        print(json.dumps(verified_result, ensure_ascii=False, indent=2))
finally:
    print("Step A — Gemini + SQLite:", json.dumps(workflow.metrics, ensure_ascii=False))
    if workflow.result is None and not workflow.clarification:
        print("No verified result. Diagnostics:", json.dumps(workflow.trace, ensure_ascii=False))

## Step B — Gemini diễn đạt tiếng Việt
Chạy lại riêng cell này để đo thời gian diễn đạt; không gọi lại SQL. Kết quả không đạt evidence validation dùng bảng xác định.

In [ ]:
import json

final_answer = None
if workflow.question != question:
    raise RuntimeError("Question changed; rerun Cell A before generating a response.")
final_answer = workflow.respond()
print("Response source:", final_answer["source"])
if final_answer.get("reason"):
    print("Fallback reason:", final_answer["reason"])
print(final_answer["text"])
if final_answer.get("scope"):
    print(final_answer["scope"])
print("Step B — Gemini response:", json.dumps(workflow.metrics, ensure_ascii=False))